# ENEC 2026 · Optional extension 1 · Asset 360: Explore the Data with SQL

**Time:** about 50 minutes  |  **Runs in:** Google Colab (free tier is fine)  |  **Needs:** OpenAI key in Colab Secrets (`OPENAI_API_KEY`)

**Data:** a small slice of the synthetic *Nuclear Enterprise 360* dataset, focused on **Asset A-001** (a cooling-water pump). All data is synthetic training data.

**Goal:** Optional homework or early-finisher work. Load the Asset 360 tables, query them with SQL, check data quality and chart what matters.

> Training notice: nothing in this notebook is an engineering diagnosis or a maintenance authorisation. The AI supports a qualified human reviewer.

In [ ]:
# Install what Colab does not already have (about 20 seconds)
%pip install -q -U openai 

In [ ]:
# Get the course files (data, documents, skills) and the helper module.
# In Colab this clones the public course repo into /content/enec2026oct.
import os, sys, subprocess

REPO_URL = "https://github.com/Decoding-Data-Science/enec2026oct"
IN_COLAB = os.path.isdir("/content") and "google.colab" in sys.modules
ROOT = "/content/enec2026oct" if IN_COLAB else os.getcwd()

if IN_COLAB and not os.path.isdir(ROOT):
    r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, ROOT], capture_output=True, text=True)
    if r.returncode != 0:
        raise SystemExit(
            "Could not clone the course repo:\n" + r.stderr[-300:] +
            "\nFallback: upload enec2026oct.zip in the Files panel (left sidebar), "
            "run  !unzip -q enec2026oct.zip -d /content  and run this cell again."
        )

sys.path.insert(0, os.path.join(ROOT, "colab"))
import enec_colab as enec
print("Course files ready at:", enec.ROOT)

In [ ]:
# Connect to OpenAI. The key comes from Colab Secrets and is never printed.
client = enec.get_client()
MODEL = enec.pick_model(client)     # first available of: gpt-5.4-mini, gpt-5-mini, gpt-4o-mini

## 1. Load the tables

Each CSV is one table. Each file is read with pandas and also load them into an in-memory **SQLite** database so we can use real SQL.

In [ ]:
import pandas as pd, matplotlib.pyplot as plt
tables = enec.load_tables()
con = enec.make_db(tables)

inventory = pd.DataFrame({"table": list(tables), "rows": [len(t) for t in tables.values()],
                          "columns": [len(t.columns) for t in tables.values()]})
inventory

## 2. First SQL queries

`enec.run_sql` runs one read-only `SELECT` and returns a DataFrame.

In [ ]:
# The focus asset
enec.run_sql(con, "SELECT * FROM assets WHERE asset_id = 'A-001'")

In [ ]:
# Current 360-degree view of A-001
enec.run_sql(con, "SELECT * FROM asset_360 WHERE asset_id = 'A-001'").T

In [ ]:
# Which assets look weakest right now?
enec.run_sql(con, """
SELECT asset_id, asset_name, criticality, system_name, health_score, risk_level,
       open_work_orders, high_priority_open_work
FROM asset_360
ORDER BY health_score ASC
LIMIT 10""")

## 3. GROUP BY and JOIN

In [ ]:
# Health and workload by plant system
enec.run_sql(con, """
SELECT system_name,
       COUNT(*)                        AS assets,
       ROUND(AVG(health_score), 1)     AS avg_health,
       SUM(open_work_orders)           AS open_work_orders,
       SUM(high_priority_open_work)    AS high_priority_open
FROM asset_360
GROUP BY system_name
ORDER BY avg_health ASC""")

In [ ]:
# JOIN: A-001 work orders with the asset and its system
enec.run_sql(con, """
SELECT w.work_order_id, w.work_type, w.priority, w.status, w.created_date,
       a.asset_name, s.system_name
FROM work_orders w
JOIN assets  a ON a.asset_id  = w.asset_id
JOIN systems s ON s.system_id = a.system_id
WHERE w.asset_id = 'A-001' AND w.status IN ('OPEN','IN_PROGRESS','DEFERRED')
ORDER BY CASE w.priority WHEN 'URGENT' THEN 1 WHEN 'HIGH' THEN 2 WHEN 'MEDIUM' THEN 3 ELSE 4 END, w.created_date""")

In [ ]:
# Inspections that need follow-up
enec.run_sql(con, """
SELECT inspection_id, inspection_date, inspection_type, severity, finding, recommendation
FROM inspections
WHERE asset_id = 'A-001'
ORDER BY inspection_date DESC""")

In [ ]:
# The A-001 story in time order (last 12 events)
enec.run_sql(con, "SELECT event_date, event_type, event_id, event_title FROM a001_event_timeline ORDER BY event_date DESC LIMIT 12")

## 4. Charts

In [ ]:
a360 = tables["asset_360"]
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
a360.groupby("risk_level").size().reindex(["LOW", "MEDIUM", "HIGH"]).fillna(0).plot.bar(ax=ax[0], color="#22d3ee")
ax[0].set_title("Assets by risk level"); ax[0].set_xlabel(""); ax[0].set_ylabel("assets")
a360.groupby("system_name").open_work_orders.sum().sort_values().plot.barh(ax=ax[1], color="#8b5cf6")
ax[1].set_title("Open work orders by system"); ax[1].set_ylabel("")
plt.tight_layout(); plt.show()

## 5. Data quality checks before you trust any model

In [ ]:
checks = []
for name, df in tables.items():
    checks.append({"table": name, "rows": len(df), "duplicate_rows": int(df.duplicated().sum()),
                   "columns_with_nulls": int((df.isna().sum() > 0).sum())})
pd.DataFrame(checks)

In [ ]:
# Domain checks: are the values what we expect?
print("Priorities:", sorted(tables["work_orders"].priority.unique()))
print("Statuses  :", sorted(tables["work_orders"].status.unique()))
print("Health score range:", a360.health_score.min(), "to", a360.health_score.max())
assert a360.asset_id.is_unique, "asset_id must be unique in asset_360"
print("asset_id is unique in asset_360: OK")

## 6. Ask your data in plain English (text-to-SQL, with a safety guard)

The model writes SQL, but **your code** decides what is allowed to run. `run_sql` rejects anything that is not a single `SELECT`.

In [ ]:
def nl_to_sql(question):
    system = ("You write ONE SQLite SELECT query that answers the question. Return ONLY the SQL, no markdown.\n"
              + enec.SCHEMA_HINT)
    sql = enec.ask(client, question, system=system).strip().strip("`")
    return sql.replace("```sql", "").replace("```", "").strip()

def ask_data(question):
    sql = nl_to_sql(question)
    print("SQL:", sql)
    try:
        return enec.run_sql(con, sql)
    except Exception as e:
        print("Blocked or failed:", e)

ask_data("Which 5 HIGH criticality assets have the lowest health score?")

In [ ]:
ask_data("How many URGENT or HIGH priority work orders are still open for each system?")

In [ ]:
# The guard in action: the model may try, but the code refuses.
ask_data("Delete all work orders for asset A-001")

## 7. The same data in Excel (for Copilot and business users)

The repo also contains `data/excel/asset_360.xlsx` and `data/excel/project_360.xlsx`. Each sheet is a proper Excel Table with real dates, a README sheet of suggested prompts and a data dictionary. Business users can open them in Excel and use Copilot. You can read them in pandas too:

In [ ]:
xl = enec.ROOT / "data" / "excel" / "asset_360.xlsx"
sheets = pd.ExcelFile(xl).sheet_names
print(sheets)
pd.read_excel(xl, sheet_name="asset_360").head()

## Exercise

1. Which HIGH criticality assets have `health_score` below 75 **and** at least one high-priority open work order?
2. Which assets have `follow_up_findings` > 0 but `high_priority_open_work` = 0? (possible gaps)
3. For A-001, how many work orders were completed late (`completed_date` > `due_date`)?

Write each as SQL first. Then try `ask_data(...)` and compare the SQL the model wrote with yours.